<a href="https://colab.research.google.com/github/Mujhtaba-10/Models/blob/main/Advection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
   import torch
   print("GPU:", torch.cuda.is_available())
   !pip -q install folium openpyxl

GPU: False


In [33]:
%%writefile partB1_prepare.py
"""
Part B, step 1 - turn the Beijing Multi-Site Air Quality files into graph-ready arrays.

Dataset: UCI "Beijing Multi-Site Air-Quality Data" (also on Kaggle). 12 CSVs named
         PRSA_Data_<Station>_20130301-20170228.csv, hourly, columns:
         No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
Colab:   from partB1_prepare import main;  main("beijing")      # folder that holds the CSVs
CLI:     python partB1_prepare.py  path/to/folder_with_csvs
Output:  graph_data.npz
"""
import glob, os, sys
import numpy as np, pandas as pd
from sklearn.neighbors import BallTree

# The dataset has NO coordinates. These are APPROXIMATE positions of the monitoring sites
# (from memory) - check them against a map / the station list before trusting the final map.
COORDS = {
    "Aotizhongxin": (39.982, 116.397), "Changping": (40.217, 116.230),
    "Dingling": (40.292, 116.220),     "Dongsi": (39.929, 116.417),
    "Guanyuan": (39.929, 116.339),     "Gucheng": (39.914, 116.184),
    "Huairou": (40.328, 116.628),      "Nongzhanguan": (39.937, 116.461),
    "Shunyi": (40.127, 116.655),       "Tiantan": (39.886, 116.407),
    "Wanliu": (39.987, 116.287),       "Wanshouxigong": (39.878, 116.352),
}
TARGET = "PM2.5"
WEATHER = ["TEMP", "PRES", "DEWP", "RAIN"]
COMPASS = {d: i * 22.5 for i, d in enumerate(
    ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"])}
K = 5                      # neighbours per node
MAX_FILL = 3               # fill gaps of at most 3 h


def _find_csvs(folder):
    """Find the station CSVs wherever they are: any depth, unzip if needed, tolerate other file names."""
    import zipfile
    need = {"PM2.5", "station", "wd", "WSPM", "year", "month", "day", "hour"}
    def scan():
        files = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
        good = []
        for f in sorted(files):
            try:
                if need <= set(pd.read_csv(f, nrows=0).columns):
                    good.append(f)
            except Exception:
                pass
        return good
    good = scan()
    if not good:                                   # maybe the zip was never extracted
        for z in glob.glob(os.path.join(folder, "**", "*.zip"), recursive=True) + glob.glob("*.zip"):
            try:
                zipfile.ZipFile(z).extractall(os.path.join(folder, "_unzipped"))
            except Exception:
                pass
        for z in glob.glob(os.path.join(folder, "_unzipped", "**", "*.zip"), recursive=True):   # nested zip
            try:
                zipfile.ZipFile(z).extractall(os.path.join(folder, "_unzipped"))
            except Exception:
                pass
        good = scan()
    return good


def load_all(folder):
    files = _find_csvs(folder)
    if not files:
        seen = sorted(glob.glob(os.path.join(folder, "**", "*"), recursive=True))[:40]
        raise FileNotFoundError(
            f"No Beijing station CSVs found under '{folder}' (cwd={os.getcwd()}).\n"
            f"Expected columns: year, month, day, hour, PM2.5, wd, WSPM, station.\n"
            f"Files seen there: {seen if seen else 'NOTHING - the download/upload step did not put files here'}")
    print(f"found {len(files)} station files")
    frames = []
    for f in files:
        d = pd.read_csv(f)
        d["time"] = pd.to_datetime(d[["year", "month", "day", "hour"]])
        frames.append(d)
    return pd.concat(frames, ignore_index=True)


def pivot(df, col, stations):
    """-> DataFrame (time x station) on a complete hourly index."""
    p = df.pivot_table(index="time", columns="station", values=col, aggfunc="mean")
    p = p.reindex(columns=stations)
    return p.asfreq("h")


def main(folder, out="graph_data.npz"):
    df = load_all(folder)
    stations = sorted(df["station"].unique())
    missing = [s for s in stations if s not in COORDS]
    assert not missing, f"add coordinates for {missing}"
    print("stations:", stations)

    # ---- concentration + observation mask (mask uses the RAW data, before any filling)
    C_raw = pivot(df, TARGET, stations)
    C_raw[C_raw < 0] = np.nan
    mask = C_raw.notna().values.astype(np.float32)          # 1 = real measurement
    C = C_raw.interpolate(limit=MAX_FILL, limit_area="inside")
    idx = C.index

    # ---- weather: wind direction (compass text) + speed -> u, v components
    wd = df["wd"].map(COMPASS)
    df = df.assign(wd_deg=wd)
    spd = pivot(df, "WSPM", stations)
    deg = pivot(df, "wd_deg", stations)
    rad = np.deg2rad(deg)
    # meteorological wind direction = where the wind comes FROM, so flow vector is negative
    U = (-spd * np.sin(rad)).interpolate(limit=6, limit_area="inside")
    V = (-spd * np.cos(rad)).interpolate(limit=6, limit_area="inside")
    W = [pivot(df, c, stations).interpolate(limit=6, limit_area="inside") for c in WEATHER]

    # ---- stack to arrays (time, node, feature); leftover NaNs -> 0 AFTER normalisation later
    met = np.stack([w.values for w in W] + [U.values, V.values], axis=-1).astype(np.float32)
    met_names = WEATHER + ["u", "v"]

    # ---- graph: k nearest neighbours by great-circle distance
    lat = np.array([COORDS[s][0] for s in stations]); lon = np.array([COORDS[s][1] for s in stations])
    tree = BallTree(np.deg2rad(np.c_[lat, lon]), metric="haversine")
    _, nb = tree.query(np.deg2rad(np.c_[lat, lon]), k=K + 1)
    src = nb[:, 1:].reshape(-1)                              # neighbour j  (message sender)
    dst = np.repeat(np.arange(len(stations)), K)             # node i      (message receiver)
    R = 6371.0
    dy = (lat[dst] - lat[src]) * np.pi / 180 * R             # km north, j -> i
    dx = (lon[dst] - lon[src]) * np.pi / 180 * R * np.cos(np.deg2rad(lat[dst]))
    dist = np.sqrt(dx**2 + dy**2)
    geo = np.stack([dx / dist, dy / dist, dist], 1).astype(np.float32)   # unit vector + km

    np.savez(out, C=C.values.astype(np.float32), mask=mask, met=met, met_names=met_names,
             times=idx.values.astype("datetime64[h]"), stations=stations, lat=lat, lon=lon,
             edge_index=np.stack([src, dst]), edge_geo=geo)
    print(f"C {C.shape}, met {met.shape}, edges {len(src)}")
    print(f"observed fraction {mask.mean():.2%}; remaining NaN in C {np.isnan(C.values).mean():.2%}")
    print(f"mean edge length {dist.mean():.1f} km")
    print("saved", out)



def _cli_arg(default, suffix=None):
    """First command-line argument that is a real argument. Ignores the '-f kernel.json'
    flags that Jupyter/Colab injects into sys.argv (that was the '-f' folder bug)."""
    import sys
    for a in sys.argv[1:]:
        if a.startswith("-") or a.endswith(".json"):
            continue
        if suffix is None or a.endswith(suffix):
            return a
    return default


if __name__ == "__main__" and "__file__" in globals():   # not when pasted into a notebook cell
    main(_cli_arg("."))

Writing partB1_prepare.py


In [34]:
%%writefile partB2_train_gnn.py
"""
Part B, step 2 - spatio-temporal GNN for PM2.5, plain PyTorch (no torch_geometric needed).

Input : graph_data.npz from partB1_prepare.py
Task  : last 24 h at every station (+ weather, + wind along each edge) -> next 6 h at every station
Colab : from partB2_train_gnn import run
        run()                       # all stations seen in training
        run(holdout="Dongsi")       # leave-one-station-out (the "map" test)
CLI   : python partB2_train_gnn.py [--holdout Dongsi] [--epochs 30]
"""
from types import SimpleNamespace
import numpy as np, torch, torch.nn as nn

LOOKBACK, HORIZON = 24, 6


def run(data="graph_data.npz", holdout=None, hidden=64, epochs=30, batch_size=32, lr=1e-3):
    args = SimpleNamespace(data=data, holdout=holdout, hidden=hidden, epochs=epochs, batch=batch_size, lr=lr)
    torch.manual_seed(0); np.random.seed(0)
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    # ------------------------------------------------------------------ load + normalise
    Z = np.load(args.data, allow_pickle=True)
    C, M, met = Z["C"], Z["mask"], Z["met"]                 # (T,N), (T,N), (T,N,6)
    stations = list(Z["stations"]); T, N = C.shape
    times = Z["times"].astype("datetime64[h]")
    EI, EG = Z["edge_index"], Z["edge_geo"]                 # cache once as numpy arrays
    src, dst = [torch.tensor(a, dtype=torch.long, device=dev) for a in EI]
    E = src.numel()

    hold = stations.index(args.holdout) if args.holdout else None
    n_tr, n_va = int(0.70 * T), int(0.85 * T)               # split BY TIME, never random

    Clog = np.log1p(np.nan_to_num(C, nan=0.0))              # log tames the heavy tail of PM2.5
    mu, sd = Clog[:n_tr][M[:n_tr] > 0].mean(), Clog[:n_tr][M[:n_tr] > 0].std()   # TRAIN stats only
    Cn = (Clog - mu) / sd * M                               # unobserved -> 0 (and mask says so)
    mm, ms = np.nanmean(met[:n_tr], (0, 1)), np.nanstd(met[:n_tr], (0, 1)) + 1e-6
    metn = np.nan_to_num((met - mm) / ms, nan=0.0).astype(np.float32)
    hr = (times.astype("datetime64[h]") - times.astype("datetime64[D]")).astype(int)
    tfeat = np.stack([np.sin(2*np.pi*hr/24), np.cos(2*np.pi*hr/24)], 1).astype(np.float32)   # (T,2)
    u_i, v_i = list(Z["met_names"]).index("u"), list(Z["met_names"]).index("v")
    Craw_obs = np.where(M > 0, C, np.nan)

    def sample(t):
        """t = index of first forecast hour. Returns x (N,F), edge_attr (E,3), y (N,H), m (N,H)."""
        hist = Cn[t-LOOKBACK:t].T.copy(); hmask = M[t-LOOKBACK:t].T.copy()      # (N,24)
        if hold is not None: hist[hold] = 0; hmask[hold] = 0                     # hidden station: no history
        x = np.concatenate([hist, hmask, metn[t-1], np.repeat(tfeat[t-1][None], N, 0)], 1)
        uu, vv = met[t-1, :, u_i], met[t-1, :, v_i]                              # raw m/s at sender nodes
        s = EI[0]
        wind_along = np.nan_to_num(uu[s]*EG[:, 0] + vv[s]*EG[:, 1]) / 3.0
        ea = np.stack([EG[:, 0], EG[:, 1], wind_along, EG[:, 2]/20.0], 1)
        y = Cn[t:t+HORIZON].T.copy(); m = M[t:t+HORIZON].T.copy()
        return x.astype(np.float32), ea.astype(np.float32), y.astype(np.float32), m.astype(np.float32)

    def batch(ts, train_mask=False):
        xs, eas, ys, ms_ = zip(*[sample(t) for t in ts])
        X, EA, Y, MK = [torch.tensor(np.stack(a), device=dev) for a in (xs, eas, ys, ms_)]
        if train_mask and hold is not None: MK[:, hold] = 0           # never train on the hidden station
        return X, EA, Y, MK

    ts_tr = np.arange(LOOKBACK, n_tr - HORIZON)
    ts_va = np.arange(n_tr, n_va - HORIZON); ts_te = np.arange(n_va, T - HORIZON)
    F = 2*LOOKBACK + metn.shape[2] + 2

    # ------------------------------------------------------------------ model
    class AdvLayer(nn.Module):
        """Message from sender j to receiver i uses both node states + edge attrs (incl. wind along edge)."""
        def __init__(self, h, ed=4):
            super().__init__()
            self.msg = nn.Sequential(nn.Linear(2*h + ed, h), nn.ReLU(), nn.Linear(h, h))
            self.upd = nn.Sequential(nn.Linear(2*h, h), nn.ReLU())
        def forward(self, z, ea):                        # z (B,N,H), ea (B,E,ed)
            m = self.msg(torch.cat([z[:, dst], z[:, src], ea], -1))          # (B,E,H)
            agg = torch.zeros_like(z).index_add_(1, dst, m) / (E / N)        # mean-ish over K neighbours
            return self.upd(torch.cat([z, agg], -1))

    class PollGNN(nn.Module):
        def __init__(self, f_in, h):
            super().__init__()
            self.enc = nn.Sequential(nn.Linear(f_in, h), nn.ReLU())
            self.l1, self.l2 = AdvLayer(h), AdvLayer(h)
            self.dec = nn.Linear(h, HORIZON)
        def forward(self, x, ea):
            z = self.enc(x); z = z + self.l1(z, ea); z = z + self.l2(z, ea)
            return self.dec(z)                                               # (B,N,H)

    def masked_mse(pred, y, m): return ((pred - y)**2 * m).sum() / m.sum().clamp(min=1)

    def evaluate(model, ts, only=None):
        """MAE/RMSE per forecast hour in ORIGINAL units (ug/m3), on observed values only."""
        model.eval(); P, Y, MK = [], [], []
        with torch.no_grad():
            for i in range(0, len(ts), 128):
                X, EA, Yb, Mb = batch(ts[i:i+128]); P.append(model(X, EA).cpu()); Y.append(Yb.cpu()); MK.append(Mb.cpu())
        P, Y, MK = torch.cat(P), torch.cat(Y), torch.cat(MK)
        if only is not None:
            keep = torch.zeros_like(MK); keep[:, only] = 1; MK = MK * keep
        inv = lambda a: np.expm1(a.numpy() * sd + mu)
        p_, y_, m_ = inv(P), inv(Y), MK.numpy()
        out = []
        for h in range(HORIZON):
            w = m_[:, :, h] > 0
            e = p_[:, :, h][w] - y_[:, :, h][w]
            out.append((np.abs(e).mean(), np.sqrt((e**2).mean())))
        return out, (P, Y, MK)

    def baselines(ts, only=None):
        """persistence (last observed hour) and, for a hidden station, inverse-distance mean of neighbours."""
        res = {"persistence": [[] for _ in range(HORIZON)], "neighbour-IDW": [[] for _ in range(HORIZON)]}
        s_np, d_np = EI[0], EI[1]; km = EG[:, 2]
        nodes = [only] if only is not None else range(N)
        for t in ts:
            for n in nodes:
                nb = s_np[d_np == n]; w = 1 / km[d_np == n]
                last_nb = Craw_obs[t-1, nb]; ok = ~np.isnan(last_nb)
                idw = (last_nb[ok] * w[ok]).sum() / w[ok].sum() if ok.any() else np.nan
                for h in range(HORIZON):
                    y = Craw_obs[t+h, n]
                    if np.isnan(y): continue
                    if not np.isnan(Craw_obs[t-1, n]) and only is None: res["persistence"][h].append(abs(Craw_obs[t-1, n] - y))
                    if not np.isnan(idw): res["neighbour-IDW"][h].append(abs(idw - y))
        return {k: [np.mean(v) if v else np.nan for v in vs] for k, vs in res.items()}

    # ------------------------------------------------------------------ train
    model = PollGNN(F, args.hidden).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)
    best, best_state, bad = 1e9, None, 0
    for ep in range(args.epochs):
        model.train(); np.random.shuffle(ts_tr); tot = 0
        for i in range(0, len(ts_tr), args.batch):
            X, EA, Y, MK = batch(ts_tr[i:i+args.batch], train_mask=True)
            loss = masked_mse(model(X, EA), Y, MK)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        va, _ = evaluate(model, ts_va, only=hold); v = np.mean([a for a, _ in va])
        print(f"epoch {ep+1:02d}  train loss {tot/ (len(ts_tr)/args.batch):.4f}  val MAE {v:.2f}")
        if v < best: best, bad, best_state = v, 0, {k: x.clone() for k, x in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 5: print("early stop"); break
    model.load_state_dict(best_state)
    torch.save({"state": best_state, "mu": mu, "sd": sd, "holdout": args.holdout}, "gnn.pt")

    # ------------------------------------------------------------------ test
    te, _ = evaluate(model, ts_te, only=hold)
    base = baselines(ts_te[::6], only=hold)                  # subsample test hours to keep it quick
    title = f"HELD-OUT station {args.holdout}" if hold is not None else "all stations"
    print(f"\nTEST ({title}), MAE in ug/m3 by forecast hour")
    print("hour      " + "  ".join(f"+{h+1}h".rjust(6) for h in range(HORIZON)))
    print("GNN       " + "  ".join(f"{a:6.1f}" for a, _ in te))
    for k, v in base.items():
        if not np.all(np.isnan(v)): print(f"{k:<10}" + "  ".join(f"{a:6.1f}" for a in v))

    # ------------------------------------------------------------------ save latest forecast for the map
    model.eval()
    with torch.no_grad():
        X, EA, _, _ = batch([ts_te[-1]])
        pr = model(X, EA)[0].cpu().numpy()                    # (N,H) normalised
    np.savez("preds.npz", pred=np.expm1(pr * sd + mu), lat=Z["lat"], lon=Z["lon"], stations=Z["stations"])
    print("saved preds.npz (forecast for the last test window, ug/m3)")
    return model


def _cli_arg(default, suffix=None):
    """First command-line argument that is a real argument. Ignores the '-f kernel.json'
    flags that Jupyter/Colab injects into sys.argv (that was the '-f' folder bug)."""
    import sys
    for a in sys.argv[1:]:
        if a.startswith("-") or a.endswith(".json"):
            continue
        if suffix is None or a.endswith(suffix):
            return a
    return default


if __name__ == "__main__" and "__file__" in globals():   # not when pasted into a notebook cell
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="graph_data.npz"); p.add_argument("--holdout", default=None)
    p.add_argument("--hidden", type=int, default=64);   p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch", type=int, default=32);    p.add_argument("--lr", type=float, default=1e-3)
    a, _ = p.parse_known_args()                         # ignores unknown flags such as Colab's -f
    run(a.data, a.holdout, a.hidden, a.epochs, a.batch, a.lr)

Writing partB2_train_gnn.py


In [24]:
!ls -la /content

total 7796
drwxr-xr-x 1 root root    4096 Oct 15 10:10 .
drwxr-xr-x 1 root root    4096 Oct 15 09:30 ..
drwxr-xr-x 4 root root    4096 Oct  2 13:50 .config
drwxr-xr-x 2 root root    4096 Oct 15 09:40 .ipynb_checkpoints
-rw-r--r-- 1 root root 7959991 Oct 15 10:10 PRSA2017_Data_20130301-20170228.zip
drwxr-xr-x 1 root root    4096 Oct  2 13:50 sample_data


In [35]:
import partB1_prepare
partB1_prepare.main(".")

found 12 station files
stations: ['Aotizhongxin', 'Changping', 'Dingling', 'Dongsi', 'Guanyuan', 'Gucheng', 'Huairou', 'Nongzhanguan', 'Shunyi', 'Tiantan', 'Wanliu', 'Wanshouxigong']
C (35064, 12), met (35064, 12, 6), edges 60
observed fraction 97.92%; remaining NaN in C 1.00%
mean edge length 16.1 km
saved graph_data.npz


In [36]:
model = run(epochs=3)

epoch 01  train loss 0.2520  val MAE 21.81
epoch 02  train loss 0.2128  val MAE 20.69
epoch 03  train loss 0.2043  val MAE 20.80

TEST (all stations), MAE in ug/m3 by forecast hour
hour         +1h     +2h     +3h     +4h     +5h     +6h
GNN         12.1    17.2    20.9    24.6    28.0    30.6
persistence  10.6    17.5    22.8    27.5    31.5    35.1
neighbour-IDW  16.0    20.0    23.8    27.8    31.2    34.7
saved preds.npz (forecast for the last test window, ug/m3)


In [37]:
   model = run(epochs=30)


epoch 01  train loss 0.2520  val MAE 21.81
epoch 02  train loss 0.2128  val MAE 20.69
epoch 03  train loss 0.2043  val MAE 20.80
epoch 04  train loss 0.1986  val MAE 20.95
epoch 05  train loss 0.1932  val MAE 20.12
epoch 06  train loss 0.1903  val MAE 20.18
epoch 07  train loss 0.1859  val MAE 20.83
epoch 08  train loss 0.1836  val MAE 19.35
epoch 09  train loss 0.1811  val MAE 19.50
epoch 10  train loss 0.1777  val MAE 20.65
epoch 11  train loss 0.1754  val MAE 19.53
epoch 12  train loss 0.1737  val MAE 19.36
epoch 13  train loss 0.1716  val MAE 19.77
early stop

TEST (all stations), MAE in ug/m3 by forecast hour
hour         +1h     +2h     +3h     +4h     +5h     +6h
GNN         11.2    16.2    19.9    23.5    26.9    30.1
persistence  10.6    17.5    22.8    27.5    31.5    35.1
neighbour-IDW  16.0    20.0    23.8    27.8    31.2    34.7
saved preds.npz (forecast for the last test window, ug/m3)


In [39]:
   run(holdout="Dongsi", epochs=30)

epoch 01  train loss 0.2496  val MAE 43.33
epoch 02  train loss 0.2099  val MAE 41.96
epoch 03  train loss 0.2013  val MAE 40.17
epoch 04  train loss 0.1951  val MAE 39.52
epoch 05  train loss 0.1897  val MAE 36.63
epoch 06  train loss 0.1867  val MAE 32.81
epoch 07  train loss 0.1826  val MAE 33.11
epoch 08  train loss 0.1807  val MAE 37.42
epoch 09  train loss 0.1784  val MAE 33.43
epoch 10  train loss 0.1744  val MAE 36.89
epoch 11  train loss 0.1722  val MAE 38.25
early stop

TEST (HELD-OUT station Dongsi), MAE in ug/m3 by forecast hour
hour         +1h     +2h     +3h     +4h     +5h     +6h
GNN         43.8    37.0    37.9    36.8    38.4    38.4
neighbour-IDW  14.9    20.6    25.8    31.1    35.7    39.7
saved preds.npz (forecast for the last test window, ug/m3)


PollGNN(
  (enc): Sequential(
    (0): Linear(in_features=56, out_features=64, bias=True)
    (1): ReLU()
  )
  (l1): AdvLayer(
    (msg): Sequential(
      (0): Linear(in_features=132, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
    (upd): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (l2): AdvLayer(
    (msg): Sequential(
      (0): Linear(in_features=132, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
    (upd): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (dec): Linear(in_features=64, out_features=6, bias=True)
)

In [40]:
import numpy as np, pandas as pd, folium
from folium.plugins import HeatMap
from IPython.display import display

P = np.load("preds.npz", allow_pickle=True)
lat, lon, names, pred = P["lat"], P["lon"], list(P["stations"]), P["pred"]   # pred: (12 stations, 6 hours)

# 1) the forecast as a table
display(pd.DataFrame(pred, index=names, columns=[f"+{h}h" for h in range(1, 7)]).round(1))

# 2) colours by US AQI category for PM2.5
BINS = [(0, 12, "#00e400"), (12, 35.4, "#ffff00"), (35.4, 55.4, "#ff7e00"),
        (55.4, 150.4, "#ff0000"), (150.4, 250.4, "#8f3f97"), (250.4, 1e9, "#7e0023")]
colour = lambda v: next(c for lo, hi, c in BINS if lo <= v < hi)

# 3) interpolate station values onto a grid (inverse-distance weighting)
def idw_grid(val, n=120, power=2.0, pad=0.05):
    glat = np.linspace(lat.min() - pad, lat.max() + pad, n)
    glon = np.linspace(lon.min() - pad, lon.max() + pad, n)
    GLon, GLat = np.meshgrid(glon, glat)
    kx = 111.32 * np.cos(np.deg2rad(lat.mean()))
    d = np.sqrt(((GLat[..., None] - lat) * 111.0) ** 2 + ((GLon[..., None] - lon) * kx) ** 2) + 1e-6
    w = 1.0 / d ** power
    return glat, glon, (w * val).sum(-1) / w.sum(-1)

# 4) draw the map for forecast hour h (0 = +1h, 5 = +6h)
def draw(h):
    val = pred[:, h]
    glat, glon, G = idw_grid(val)
    m = folium.Map(location=[lat.mean(), lon.mean()], zoom_start=10, tiles="cartodbpositron")
    HeatMap([[a, b, g] for a, row in zip(glat, G) for b, g in zip(glon, row)],
            radius=14, blur=18, min_opacity=0.35).add_to(m)
    for a, b, v, nm in zip(lat, lon, val, names):
        folium.CircleMarker([a, b], radius=8, color="black", weight=1, fill=True,
                            fill_color=colour(v), fill_opacity=1,
                            tooltip=f"{nm}: {v:.0f} ug/m3 (+{h+1}h)").add_to(m)
    m.save(f"map_h{h+1}.html")
    return m

draw(0)    # +1h map: this displays below the cell

,+1h,+2h,+3h,+4h,+5h,+6h
Aotizhongxin,9.0,10.1,12.600000,15.700000,19.900000,24.799999
Changping,13.1,13.7,13.500000,14.000000,15.100000,14.700000
Dingling,12.7,13.6,13.700000,13.600000,14.700000,13.900000
Dongsi,5.5,4.9,6.800000,9.800000,11.800000,10.800000
Guanyuan,9.2,14.0,16.700001,20.600000,25.799999,29.500000
Gucheng,11.4,11.2,11.600000,12.700000,14.300000,15.700000
Huairou,10.0,12.1,14.100000,16.200001,18.000000,19.200001
Nongzhanguan,8.4,12.9,17.900000,20.500000,27.700001,31.200001
Shunyi,8.9,10.3,13.600000,17.200001,22.299999,26.700001
Tiantan,9.7,11.6,14.700000,18.100000,22.799999,27.600000


/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
